# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. You'll see how data defined by a [Croissant schema](https://mlcommons.org/croissant/) can be inspected and processed end-to-end in pandas for further analysis or modeling.

### Dataset Source
- **Croissant schema URL:** [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)
- **Package DOI:** [10.71728/senscience.vcs2-05nj](https://sen.science/doi/10.71728/senscience.vcs2-05nj)


In [ ]:
# Ensure `mlcroissant` and dependencies are installed
!pip install mlcroissant --quiet


## 1. Data Loading
Load metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# View the main metadata fields
md = dataset.metadata
print(f"Dataset Name: {md.name}")
print(f"Description: {md.description}")
print(f"Number of record sets: {len(md.record_sets)}")
if hasattr(md, 'keywords'):
    print(f"Keywords: {md.keywords}")

## 2. Data Overview
Review available record sets, their fields, and their `@id`s.

Each record set may represent a major table or entity, and each has fields (`cr:Field`)—columns, typically—each with an `@id`. We print their structure below.

In [ ]:
# Show all record sets and their fields by @id
print('Record Sets:')
record_sets = list(dataset.metadata.record_sets)
for i, rs in enumerate(record_sets):
    print(f"[{i}] Record set name: {rs.name} -- @id: {rs.id}")
    print("    Fields:")
    for f in rs.fields:
        print(f"      - {f.name} (field @id: {f.id}) | dataType: {f.data_type}")
    print()
# Pick the first record set for exploration below
if record_sets:
    main_record_set_id = record_sets[0].id
    print(f"Using record set: {record_sets[0].name} (@id: {main_record_set_id}) for analysis.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll extract all data tables (record sets) by their `@id`, and show the first rows of the main record set table.

In [ ]:
# Pull all record sets into DataFrames
dataframes = {}
for rs in dataset.metadata.record_sets:
    print(f"Reading data for record set: {rs.name} (@id: {rs.id})")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print("   Shape:", df.shape)
    print("   Columns:", list(df.columns))

# For this notebook, we'll use the first record set extracted above
main_rs_id = list(dataframes.keys())[0]
main_df = dataframes[main_rs_id]
print(f"\nPreview of main record set ({main_rs_id}):")
display(main_df.head())

## 4. Exploratory Data Analysis (EDA)

Let's apply some exploratory data analysis to numeric fields, using their `@id` as identifiers. For illustration, we'll pick a field such as `phq9_total_score` (if present) or any available numeric field.

- **Filtering:** Remove records with unusually high/low values (potential outliers)
- **Normalizing:** Create a z-score normalized version of the field
- **Grouping:** Aggregate statistics by a chosen categorical field, e.g., `gender` if it exists

In [ ]:
"""
Choose numeric_field_id and group_field_id by inspecting available columns and field @id above.
"""
# Look for numeric and categorical fields
numeric_col_candidates = [col for col in main_df.columns if main_df[col].dtype in (int, float, 'int64', 'float64') or main_df[col].apply(lambda x: isinstance(x, (int, float))).all()]
if not numeric_col_candidates and len(main_df.columns) > 0:
    # Try to convert likely numeric columns
    for col in main_df.columns:
        try:
            main_df[col] = pd.to_numeric(main_df[col])
        except:
            pass
    numeric_col_candidates = [col for col in main_df.columns if main_df[col].dtype in (int, float, 'int64', 'float64')]

if numeric_col_candidates:
    numeric_field_id = numeric_col_candidates[0]  # First numeric field
    print(f"Using numeric field: {numeric_field_id}")

    # Threshold for basic filtering (use as an example, adjust as needed)
    threshold = main_df[numeric_field_id].mean() + main_df[numeric_field_id].std()

    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else 1)
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a categorical field if present
    group_candidates = [c for c in main_df.columns if main_df[c].dtype == object and c != numeric_field_id]
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field}:")
        display(grouped_df)
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and relationship with the group field using matplotlib or seaborn. Plots help illuminate patterns and outliers.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field_id' in locals() and numeric_col_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'group_field' in locals():
        plt.figure(figsize=(8,4))
        sns.boxplot(
            x=main_df[group_field],
            y=main_df[numeric_field_id],
            showmeans=True
        )
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=30, ha='right')
        plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion

- In this notebook, we've loaded the FAIR² Mental Health Survey Dataset via its Croissant schema, explored available record sets and fields by their `@id`, and demonstrated standard data processing using `mlcroissant`. 
- The dataset enables research into mental health indicators in Kilifi County, Kenya, and is AI-ready.
- Further, more advanced analysis can be performed using the record set structure and field definitions supplied by the Croissant standard.
